# 기업

In [ ]:
import pandas as pd

df = pd.read_csv(r"22번 스코어 산출\통합_스코어_데이터.csv")

df_2023 = df[df["연도"] == 2023]
valid_companies = df_2023[df_2023["부실라벨_ICR3년"] == 0]["사업자등록번호"].unique()

filtered_df = df[
    df["사업자등록번호"].isin(valid_companies) & 
    df["연도"].isin([2022, 2023, 2024])
]

filtered_df.to_csv(r"23번. 대시보드\통합_스코어_데이터_필터링.csv", index=False)

In [3]:
filtered_df.shape

(11271, 14)

In [4]:
filtered_df['사업자등록번호'].value_counts()

사업자등록번호
8988800759    3
1018116269    3
1018117953    3
1018118781    3
1018118985    3
             ..
1058161048    2
1018680937    2
1048639136    2
8718601803    2
1298600585    1
Name: count, Length: 3829, dtype: int64

In [5]:
import os

folder_name = os.path.basename(os.getcwd())
print(folder_name)

B_도매및소매업


In [ ]:
# ============================================================
# Top10 1위 조합 기반 SHAP 분석 -> 통합_스코어_데이터_필터링.csv 에
#   (사업자등록번호, 연도) 기준으로 Top10 변수의 Value/SHAP_Value +
#   base_value / pred_prob / pred_label / threshold 컬럼 추가
# ============================================================

import os
import platform
import warnings
import numpy as np
import pandas as pd
import shap
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score
from xgboost import XGBClassifier

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")


# ============================================================
# 1. Top10_우수모델.csv 1번째 행 정보 로드
# ============================================================
TOP10_PATH = "15번. 우수모델 데이터/Top10 우수모델.csv"

top10 = pd.read_csv(TOP10_PATH, index_col=0)
row = top10.iloc[0]

FEATURE_SET  = row["FeatureSet"]
FEATURE_FILE = row["FeatureFile"]
METHOD       = row["Method"]
SMOTE_RATIO  = row["SMOTE_Ratio"]
MODEL_NAME   = row["Model"]

print("=" * 70)
print("Top10 1위 조합")
print(f"  FeatureSet  : {FEATURE_SET}")
print(f"  FeatureFile : {FEATURE_FILE}")
print(f"  Method      : {METHOD}")
print(f"  SMOTE_Ratio : {SMOTE_RATIO}")
print(f"  Model       : {MODEL_NAME}")
print("=" * 70)


# ============================================================
# 2. 경로 / 설정값
# ============================================================
TRAIN_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH    = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = os.path.join(r'13번.피처셀렉션\M19_도매_소매업', FEATURE_FILE)

TARGET_COL   = "부실라벨_ICR3년"
YEAR_COL     = "회계년도"
COMPANY_COL  = "회사명"
RANDOM_STATE = 42
RECALL_MIN   = 0.9

N_TOP        = 10  # abs_SHAP 기준 상위 N개


# ============================================================
# 3. 불균형 처리 함수 (기존과 동일)
# ============================================================
def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    minority   = X[y == 1].copy()
    n_minority = len(minority)
    n_majority = (y == 0).sum()
    target_n = int(n_majority * ratio) if ratio else n_majority
    n_synth  = max(0, target_n - n_minority)
    if n_synth == 0 or n_minority < 5:
        return X, y
    ctgan = CTGAN(epochs=300, verbose=False)
    ctgan.fit(minority)
    synth = ctgan.sample(n_synth)
    synth = synth[X.columns]
    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    method_l = str(method).strip().lower()
    ratio = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("BorderlineSMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("SMOTE 사용을 위해 'pip install imbalanced-learn'이 필요합니다.")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    print(f"  [경고] 알 수 없는 Method='{method}' -> ClassWeight(pos_weight)로 처리합니다.")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


def find_threshold_at_recall(y_true, y_prob, recall_min=RECALL_MIN):
    thresholds = np.arange(0.01, 1.0, 0.01)
    valid = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        rec = recall_score(y_true, y_pred, zero_division=0)
        if rec >= recall_min:
            valid.append(round(thr, 2))
    if not valid:
        return None
    return max(valid)


# ============================================================
# 4. 데이터 로드 + 모델 학습 (기존과 동일)
# ============================================================
train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)
y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]
all_data     = pd.concat([train_full, test], ignore_index=True)

feat_df      = pd.read_csv(FEATURE_PATH)
col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

print(f"\n피처 수    : {len(use_features)}개")
print(f"전체 데이터: {len(all_data)}행  |  기업 수: {all_data[COMPANY_COL].nunique()}개")
print("=" * 70)

imputer     = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(train_full[use_features]),
    columns=use_features
)
X_test_imp  = pd.DataFrame(
    imputer.transform(test[use_features]),
    columns=use_features
)

X_train_res, y_train_res, pos_weight = apply_resampling(
    X_train_imp, y_train_full, METHOD, SMOTE_RATIO
)
print(f"  리샘플링 적용({METHOD}, SMOTE_Ratio={SMOTE_RATIO}): "
      f"{len(X_train_imp)}행 -> {len(X_train_res)}행, pos_weight={pos_weight:.4f}")

model = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="aucpr",
    random_state=RANDOM_STATE, verbosity=0,
    scale_pos_weight=pos_weight
)
model.fit(X_train_res, y_train_res)

y_prob_test = model.predict_proba(X_test_imp)[:, 1]
found_threshold = find_threshold_at_recall(y_test, y_prob_test, RECALL_MIN)
THRESHOLD = found_threshold if found_threshold is not None else 0.01
print(f"  Test 기준 threshold (Recall >= {RECALL_MIN}) = {THRESHOLD:.2f}")


# ============================================================
# 5. 전체 데이터(all_data)에 대해 한 번에 SHAP 계산
# ============================================================
all_feat_imp = pd.DataFrame(
    imputer.transform(all_data[use_features]),
    columns=use_features,
    index=all_data.index
)

explainer = shap.TreeExplainer(model)

print("\nSHAP 값 계산 중... (전체 데이터 대상, 다소 시간이 걸릴 수 있음)")
sv_all = explainer(all_feat_imp[use_features])
print("SHAP 값 계산 완료")

y_prob_all = model.predict_proba(all_feat_imp[use_features])[:, 1]
y_pred_all = (y_prob_all >= THRESHOLD).astype(int)

feature_arr = np.array(use_features)


# ============================================================
# 6. 각 행(all_data 기준)에 대해 abs_SHAP 상위 N개 추출
#    -> Top1~Top10 Feature / Value / SHAP_Value 컬럼 생성
# ============================================================
n_rows = len(all_data)
top_feature = np.empty((n_rows, N_TOP), dtype=object)
top_value   = np.full((n_rows, N_TOP), np.nan)
top_shap    = np.full((n_rows, N_TOP), np.nan)

for i in range(n_rows):
    shap_vals = sv_all.values[i]
    order = np.argsort(-np.abs(shap_vals))[:N_TOP]  # abs_SHAP 내림차순 상위 N개 인덱스

    k = len(order)
    top_feature[i, :k] = feature_arr[order]
    top_value[i, :k]   = all_feat_imp.iloc[i][feature_arr[order]].values
    top_shap[i, :k]    = shap_vals[order]

shap_summary = pd.DataFrame({
    COMPANY_COL: all_data[COMPANY_COL],
    YEAR_COL:    all_data[YEAR_COL],
    "base_value": sv_all.base_values,
    "threshold":  THRESHOLD,
    "pred_prob":  y_prob_all,
    "pred_label": y_pred_all,
})

for r in range(N_TOP):
    shap_summary[f"Top{r+1}_Feature"]    = top_feature[:, r]
    shap_summary[f"Top{r+1}_Value"]      = top_value[:, r]
    shap_summary[f"Top{r+1}_SHAP_Value"] = top_shap[:, r]

# 동일 (회사명, 회계년도) 중복행이 있으면 첫 행만 사용 (doc4의 처리방식과 동일)
shap_summary = shap_summary.drop_duplicates(subset=[COMPANY_COL, YEAR_COL], keep="first")


# ============================================================
# 7. filtered_df 와 병합
#    매칭 키: (회사명, 연도) <-> (회사명, 회계년도)
#    ※ all_data에 '사업자등록번호'가 있다면 이걸로 매칭하는 게 더 안전합니다.
#       있다면 아래 merge의 on= 부분을 ["사업자등록번호","연도"]로 바꾸세요.
# ============================================================
filtered_df = pd.read_csv("23번. 대시보드\통합_스코어_데이터_필터링.csv")

merged = filtered_df.merge(
    shap_summary,
    left_on=["회사명", "연도"],
    right_on=[COMPANY_COL, YEAR_COL],
    how="left",
)

# 중복된 키 컬럼 정리
if COMPANY_COL != "회사명" or YEAR_COL != "연도":
    merged = merged.drop(columns=[c for c in [COMPANY_COL, YEAR_COL] if c not in ["회사명", "연도"]])

n_matched   = merged["pred_prob"].notna().sum()
n_unmatched = merged["pred_prob"].isna().sum()
print(f"\n매칭된 행: {n_matched} / 매칭 안 된 행: {n_unmatched}")

merged.to_csv(
    f"..\\대시보드\\Data\\{folder_name}_기업.csv",
    index=False,
    encoding="utf-8-sig"
)


Top10 1위 조합
  FeatureSet  : top65_dedup52
  FeatureFile : lasso_features_top65--52.csv
  Method      : ClassWeight
  SMOTE_Ratio : -
  Model       : XGBoost

피처 수    : 52개
전체 데이터: 39908행  |  기업 수: 6106개
  리샘플링 적용(ClassWeight, SMOTE_Ratio=-): 28111행 -> 28111행, pos_weight=25.6455
  Test 기준 threshold (Recall >= 0.9) = 0.34

SHAP 값 계산 중... (전체 데이터 대상, 다소 시간이 걸릴 수 있음)
SHAP 값 계산 완료

매칭된 행: 11271 / 매칭 안 된 행: 0


# 산업 SHAP

In [6]:
import pandas as pd



# ============================================================
# feature_analysis CSV 로드
# ============================================================
df = pd.read_csv(
    rf"16번. SHAP\전역\{FEATURE_SET}\feature_analysis_{FEATURE_SET}.csv"
)

# ============================================================
# 핵심피처(Combined_Rank, Feature, Category, Feature_Type)만 추출
# ============================================================
core_df = df[df["Feature_Type"] == "★ 핵심피처 (SHAP+Perm 모두 높음)"][
    ["Combined_Rank", "Feature", "Category", "Feature_Type"]
].reset_index(drop=True)

print(f"핵심피처 {len(core_df)}개 추출")
print(core_df.to_string(index=False))

# ============================================================
# 저장
# ============================================================
out_path = rf"..\대시보드\Data\{folder_name}_산업.csv"
core_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {out_path}")

핵심피처 9개 추출
 Combined_Rank                 Feature            Category             Feature_Type
             1           총자본영업이익률_diff 수익성 (Profitability) ★ 핵심피처 (SHAP+Perm 모두 높음)
             2             영업이익률_ratio                 미분류 ★ 핵심피처 (SHAP+Perm 모두 높음)
             2                   ROA변화 수익성 (Profitability) ★ 핵심피처 (SHAP+Perm 모두 높음)
             4                 금융비용부담률 수익성 (Profitability) ★ 핵심피처 (SHAP+Perm 모두 높음)
             5                   자본잠식률      안정성 (Solvency) ★ 핵심피처 (SHAP+Perm 모두 높음)
             6 금융비용대매출액_ratio_industry                 미분류 ★ 핵심피처 (SHAP+Perm 모두 높음)
             7   매출액순이익률_diff_industry 수익성 (Profitability) ★ 핵심피처 (SHAP+Perm 모두 높음)
             8                  매출액증가율        성장성 (Growth) ★ 핵심피처 (SHAP+Perm 모두 높음)
             9          영업CF_유동부채_diff    현금흐름 (Cash Flow) ★ 핵심피처 (SHAP+Perm 모두 높음)

저장 완료: ..\대시보드\Data\B_도매및소매업_산업.csv
